# Scraping database review

This notebook browses `scraping.db`, the scraping module's only implemented database. The top-level `src/storage/` module is still a skeleton, so it has no schema or data to review.

> **Kernel note:** select the project's `.venv` Python interpreter/kernel in VS Code so `pandas` and `src.scraping` imports resolve.

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DB_PATH = REPO_ROOT / 'scraping.db'
print(f'Repository: {REPO_ROOT}')
print(f'Database:   {DB_PATH} ({DB_PATH.stat().st_size if DB_PATH.exists() else 0:,} bytes)')

Repository: /Users/kumo/programming/competitor_product_search
Database:   /Users/kumo/programming/competitor_product_search/scraping.db (12,881,920 bytes)


In [ ]:
from src.scraping.storage import ScrapeDB

db = ScrapeDB(DB_PATH)
db.init_db()  # Idempotent: creates the six tables on a fresh, empty database.

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    db.conn,
)
table_summary = pd.DataFrame(
    [
        {'table': name, 'rows': db.conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]}
        for name in tables['name']
    ]
)
display(table_summary)

,table,rows
0,escalations,0
1,golden_samples,14
2,invalid_target_phrases,0
3,parsers,2
4,results,8
5,scrape_runs,5


## `scrape_runs`

The operational log: every scraper execution (including Direct API routes), its outcome and latency, plus `repair_model` only when an LLM repair ran.

In [3]:
runs = pd.read_sql_query('SELECT * FROM scrape_runs ORDER BY id DESC', db.conn)
display(runs)

runs_with_parser = pd.read_sql_query(
    """
    SELECT r.*, p.site AS parser_site, p.version AS parser_version,
           e.reason AS escalation_reason, e.signature AS escalation_signature
    FROM scrape_runs AS r
    LEFT JOIN parsers AS p ON p.id = r.winning_parser_id
    LEFT JOIN escalations AS e ON e.id = r.escalation_id
    ORDER BY r.id DESC
    """,
    db.conn,
)
# display(runs_with_parser)


,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,repair_model,latency_ms,signature,error,escalation_id
0,6,https://www.argos.co.uk/product/3284476,www.argos.co.uk,argos,ArgosScraper,2026-08-21T20:58:33.742Z,success,fast,4.0,1,None,42315,None,None,None
1,5,https://www.tesco.com/shop/en-GB/products/325439180,www.tesco.com,tesco,TescoScraper,2026-08-21T20:25:13.668Z,success,fast,3.0,1,None,10153,None,None,None
2,4,https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3,www.amazon.co.uk,amazon,AmazonUKScraper,2026-08-21T19:25:58.358Z,success,fast,NaN,1,None,17011,None,None,None
3,3,https://www.tesco.com/shop/en-GB/products/325439177,www.tesco.com,tesco,TescoScraper,2026-08-21T18:26:22.917Z,success,fast,3.0,1,None,14383,None,None,None
4,2,https://www.argos.co.uk/product/7726851,www.argos.co.uk,argos,ArgosScraper,2026-08-21T18:25:05.968Z,success,fast,4.0,1,None,84544,None,None,None


## `results`

This is the main scraped-product data table. `id` is this table's primary key; `run_id` is the only direct link to `scrape_runs.id`. The overview keeps JSON compact, and the optional second view expands `product_data`.

In [4]:
def shorten(value: object, limit: int = 180) -> object:
    if value is None or pd.isna(value):
        return value
    text = str(value)
    return text if len(text) <= limit else text[:limit] + ' …'


def decode_json(value: object) -> dict:
    if not value:
        return {}
    try:
        decoded = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {'_unparseable_product_data': value}
    return decoded if isinstance(decoded, dict) else {'_json_value': decoded}


results = pd.read_sql_query('SELECT * FROM results ORDER BY id DESC', db.conn)
results_overview = results.copy()
if 'product_data' in results_overview:
    results_overview['product_data'] = results_overview['product_data'].map(shorten)
display(results_overview)

# `id` is the results-table primary key; `run_id` is the only link to scrape_runs.id.
result_metadata = results.reindex(columns=['id', 'url', 'site', 'scraped_at', 'run_id'])
result_fields = pd.json_normalize(results['product_data'].map(decode_json).tolist())
# results_flat = result_metadata.join(result_fields)
# display(result_fields)

,id,url,site,scraped_at,product_data,run_id
0,9,https://www.argos.co.uk/product/3284476,argos,2026-08-21T20:58:33.616167+00:00,"{""url"":""https://www.argos.co.uk/product/3284476"",""website"":""argos"",""scraped_at"":""2026-08-21T20:58:33.616167Z"",""source_type"":""html"",""parser_version"":""cs_2026...",6.0
1,8,https://www.argos.co.uk/product/3284476,argos,2026-08-21T20:54:06.612747+00:00,"{""url"":""https://www.argos.co.uk/product/3284476"",""website"":""argos"",""scraped_at"":""2026-08-21T20:54:06.612747Z"",""source_type"":""html"",""parser_version"":""cs_2026...",NaN
2,7,https://www.tesco.com/shop/en-GB/products/325439180,tesco,2026-08-21T20:52:58.147498+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/325439180"",""website"":""tesco"",""scraped_at"":""2026-08-21T20:52:58.147498Z"",""source_type"":""html"",""parser_versi...",NaN
3,6,https://www.tesco.com/shop/en-GB/products/325439180,tesco,2026-08-21T20:25:13.634468+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/325439180"",""website"":""tesco"",""scraped_at"":""2026-08-21T20:25:13.634468Z"",""source_type"":""html"",""parser_versi...",NaN
4,5,https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3?th=1&psc=1&language=en_GB&currency=GBP,amazon,2026-08-21T20:24:49.331482+00:00,"{""url"":""https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3?th=1&psc=1&language=en_GB&currency=GBP"",""website"":""amazon"",""scraped_at"":""2026...",NaN
5,4,https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3?th=1&psc=1&language=en_GB&currency=GBP,amazon,2026-08-21T19:25:58.355802+00:00,"{""url"":""https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3?th=1&psc=1&language=en_GB&currency=GBP"",""website"":""amazon"",""scraped_at"":""2026...",NaN
6,3,https://www.tesco.com/shop/en-GB/products/325439177,tesco,2026-08-21T18:26:22.892494+00:00,"{""url"":""https://www.tesco.com/shop/en-GB/products/325439177"",""website"":""tesco"",""scraped_at"":""2026-08-21T18:26:22.892494Z"",""source_type"":""html"",""parser_versi...",NaN
7,2,https://www.argos.co.uk/product/7726851,argos,2026-08-21T18:25:05.464768+00:00,"{""url"":""https://www.argos.co.uk/product/7726851"",""website"":""argos"",""scraped_at"":""2026-08-21T18:25:05.464768Z"",""source_type"":""html"",""parser_version"":""cs_2026...",NaN


## `parsers`

Generated parser code is intentionally shortened here. Use `show_blob` below to inspect a full parser.

In [5]:
parsers = pd.read_sql_query('SELECT * FROM parsers ORDER BY id DESC', db.conn)
parsers_overview = parsers.copy()
if 'code' in parsers_overview:
    parsers_overview['code'] = parsers_overview['code'].map(shorten)
display(parsers_overview)

,id,site,version,code,page_type_scope,status,created_at,created_by
0,4,argos,cs_20260818_201425,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n soup = BeautifulSoup(html, 'lxml')\n \n # Check ...",None,active,2026-08-18T20:14:25.960Z,initial
1,3,tesco,cs_20260818_200257,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n # Non-product page detection\n path = url.split('?'...",None,active,2026-08-18T20:02:57.330Z,initial


## `golden_samples`

Golden snapshots validate newly repaired parsers. The flattened view makes expected ProductData fields easy to compare.

In [6]:
goldens = pd.read_sql_query('SELECT * FROM golden_samples ORDER BY id DESC', db.conn)
goldens_overview = goldens.copy()
for column in ['html_snapshot', 'expected_output']:
    if column in goldens_overview:
        goldens_overview[column] = goldens_overview[column].map(shorten)
display(goldens_overview)

golden_metadata = goldens.reindex(columns=['id', 'site', 'page_type', 'captured_at', 'is_stale'])
golden_fields = pd.json_normalize(goldens['expected_output'].map(decode_json).tolist())
goldens_flat = golden_metadata.join(golden_fields)
display(goldens_flat)

,id,site,page_type,html_snapshot,expected_output,captured_at,is_stale,created_by
0,20,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/3284476"", ""website"": ""argos"", ""scraped_at"": ""2026-08-21T20:54:06.612747Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-21T20:54:06.704Z,0,auto
1,19,tesco,standard,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.08.19-07ae612a"",""mfe-analytics"":""2026.08.12-c8505ff2"",""mfe-basket-manager"":""2026.08.11-53089089"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325439180"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-21T20:25:13.634468Z"", ""source_type"": ""html"", ""pars...",2026-08-21T20:25:13.676Z,0,auto
2,18,tesco,discounted,"<!doctype html>\n<html lang=""en-GB""><head>\n <meta charset=""UTF-8"">\n <meta name=""viewport"" content=""width=device-width, initial-scale=1.0"">\n <met...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325439177"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-21T18:26:22.892494Z"", ""source_type"": ""html"", ""pars...",2026-08-21T18:26:22.925Z,0,auto
3,17,argos,discounted,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script type=""text/javascript"" async="""" src...","{""url"": ""https://www.argos.co.uk/product/8695390"", ""website"": ""argos"", ""scraped_at"": ""2026-08-18T20:10:39.927519Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-18T20:14:26.019Z,0,coldstart
4,16,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/6793641"", ""website"": ""argos"", ""scraped_at"": ""2026-08-18T20:10:39.713730Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-18T20:14:26.008Z,0,coldstart
5,15,argos,standard,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script type=""text/javascript"" async="""" src...","{""url"": ""https://www.argos.co.uk/product/7726851"", ""website"": ""argos"", ""scraped_at"": ""2026-08-18T20:10:39.510217Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-18T20:14:26.003Z,0,coldstart
6,14,argos,discounted,"<!doctype html>\n<html lang=""en"" data-brand=""argos"" data-mode=""light"" style=""--sticky-footer-height: 0px;""><head><script src=""https://www.paypal.com/sdk/js?...","{""url"": ""https://www.argos.co.uk/product/7831935"", ""website"": ""argos"", ""scraped_at"": ""2026-08-18T20:10:39.292087Z"", ""source_type"": ""html"", ""parser_version"":...",2026-08-18T20:14:25.968Z,0,coldstart
7,13,tesco,membership,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.08.17-619f4369"",""mfe-analytics"":""2026.08.12-c8505ff2"",""mfe-basket-manager"":""2026.08.11-53089089"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/303351247"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-18T20:01:29.787636Z"", ""source_type"": ""html"", ""pars...",2026-08-18T20:02:57.355Z,0,coldstart
8,12,tesco,discounted,"<!doctype html>\n<html lang=""en-GB""><head>\n <meta charset=""UTF-8"">\n <meta name=""viewport"" content=""width=device-width, initial-scale=1.0"">\n <met...","{""url"": ""https://www.tesco.com/shop/en-GB/products/325456973"", ""website"": ""tesco"", ""scraped_at"": ""2026-08-18T20:01:29.626588Z"", ""source_type"": ""html"", ""pars...",2026-08-18T20:02:57.353Z,0,coldstart
9,11,tesco,discounted,"<!DOCTYPE html>\n<!-- VERSION: {""mfe-orchestrator"":""2026.08.17-619f4369"",""mfe-analytics"":""2026.08.12-c8505ff2"",""mfe-basket-manager"":""2026.08.11-53089089"",""m...","{""url"": ""https://www.tesco.com/shop/en-GB/products/330702828"

,id,site,page_type,captured_at,is_stale,url,website,scraped_at,source_type,parser_version,title,brand,gtin,image_urls,variant,price,currency,list_price,membership_price,in_stock,availability_raw,raw,variant.size,variant.color,variant.pack_qty
0,20,argos,standard,2026-08-21T20:54:06.704Z,0,https://www.argos.co.uk/product/3284476,argos,2026-08-21T20:54:06.612747Z,html,cs_20260818_201425,Forest Pressure Treated Wooden Shed Base - 6 x 3ft,Forest Garden,5013050000000,"[https://media.4rgos.it/i/Argos/3284476_R_Z001A, https://media.4rgos.it/i/Argos/3284476_R_Z002A, https://media.4rgos.it/i/Argos/3284476_R_Z003A, https://med...",NaN,58.00,GBP,NaN,NaN,True,Available credit options,None,NaN,NaN,NaN
1,19,tesco,standard,2026-08-21T20:25:13.676Z,0,https://www.tesco.com/shop/en-GB/products/325439180,tesco,2026-08-21T20:25:13.634468Z,html,cs_20260818_200257,KitchenAid Artisan Mixer 125 Pistachio,KitchenAid,05413184802202,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/3f8f3d68-cc56-4118-895d-a122ddb08756/543652fd96f64c669227a4ec26080e04_1752598690.jpeg, https://di...",NaN,366.95,GBP,NaN,NaN,True,In stock,None,NaN,NaN,NaN
2,18,tesco,discounted,2026-08-21T18:26:22.925Z,0,https://www.tesco.com/shop/en-GB/products/325439177,tesco,2026-08-21T18:26:22.892494Z,html,cs_20260818_200257,KitchenAid Artisan Mixer 125 Mineral Water,KitchenAid,05413184910853,"[https://digitalcontent.api.tesco.com/v2/media/marketplace/fdff8d37-2f6b-4aed-b669-332e4a296b56/jLLT4UaVBRhH7T3XCUpIWoKXY_2064789494.jpeg, https://digitalco...",NaN,367.20,GBP,549.00,NaN,True,In stock,None,NaN,NaN,NaN
3,17,argos,discounted,2026-08-18T20:14:26.019Z,0,https://www.argos.co.uk/product/8695390,argos,2026-08-18T20:10:39.927519Z,html,coldstart_v1,Samsung 55 Inch UE55U7000HKXXU Smart 4K UHD HDR Crystal TV,Samsung,8806099010003,"[https://media.4rgos.it/i/Argos/8695390_R_Z001A, https://media.4rgos.it/i/Argos/8695390_R_A001, https://media.4rgos.it/i/Argos/8695390_R_A002, https://media...",NaN,349.00,GBP,369.00,NaN,True,In stock,None,55 inch,NaN,NaN
4,16,argos,standard,2026-08-18T20:14:26.008Z,0,https://www.argos.co.uk/product/6793641,argos,2026-08-18T20:10:39.713730Z,html,coldstart_v1,Target Omni Auto Scoring System,Target,5050807079506,"[https://media.4rgos.it/i/Argos/6793641_R_Z001A, https://media.4rgos.it/i/Argos/6793641_R_Z002A, https://media.4rgos.it/i/Argos/6793641_R_Z003A, https://med...",NaN,500.00,GBP,NaN,NaN,True,In stock,None,NaN,NaN,NaN
5,15,argos,standard,2026-08-18T20:14:26.003Z,0,https://www.argos.co.uk/product/7726851,argos,2026-08-18T20:10:39.510217Z,html,coldstart_v1,PlayStation 5 Digital Edition - Slim Console,PlayStation,711719020714,"[https://media.4rgos.it/i/Argos/7726851_R_Z001A, https://media.4rgos.it/i/Argos/7726851_R_Z002A, https://media.4rgos.it/i/Argos/7726851_R_Z003A, https://med...",NaN,519.99,GBP,NaN,NaN,True,In stock,None,NaN,NaN,NaN
6,14,argos,discounted,2026-08-18T20:14:25.968Z,0,https://www.argos.co.uk/product/7831935,argos,2026-08-18T20:10:39.292087Z,html,coldstart_v1,Ninja Thirsti Black Travel Bottle - 530ml,Ninja,0622356293709,"[https://media.4rgos.it/i/Argos/7831935_R_Z001A, https://media.4rgos.it/i/Argos/7831935_R_Z002A, https://media.4rgos.it/i/Argos/7831935_R_Z003A, https://med...",NaN,24.00,GBP,30.00,NaN,True,Available credit options,None,NaN,Black,NaN
7,13,tesco,membership,2026-08-18T20:02:57.355Z,0,https://www.tesco.com/shop/en-GB/products/303351247,tesco,2026-08-18T20:01:29.787636Z,html,coldstart_v1,Warburtons Soft & Sliced Original Bagels 5 Pack,Warburtons,05010044008707,[https://digitalcontent.api.tesco.com/v2/media/ghs/f1ec19f1-9b25-4775-9edd-3818e07a8757/089527b5-c3d8-49e2-a145-a3e57f0338cb_1531756589.jpeg?h=225&w=225],NaN,1.90,GBP,NaN,1.35,True,In stock,None,NaN,NaN,5
8,12,tesco,discounted,2026-08-18T20:02:57.353Z,0,https://www.tesco.com/shop/en-GB/products/325456973,tesco,2026-08-18T20:01:29.626588Z,html,coldstart_v1,Black 3 Piece Hard Shell Suitcase Set | Black,Neo Direct,05056293917681,"[https://digitalcontent.api.tesco.com/v2/me

## `escalations`

Escalations are deduplicated by signature. The second view uses the app's store API for the currently open queue.

In [7]:
from src.scraping.storage import EscalationStore, RunStore

escalations = pd.read_sql_query('SELECT * FROM escalations ORDER BY id DESC', db.conn)
escalations_overview = escalations.copy()
if 'snapshot' in escalations_overview:
    escalations_overview['snapshot'] = escalations_overview['snapshot'].map(shorten)
display(escalations_overview)

open_escalations = pd.DataFrame(EscalationStore(db).get_open())
display(open_escalations)

# Override this id to drill into a different ticket.
ESCALATION_ID = (
    int(open_escalations.iloc[0]['id']) if not open_escalations.empty else None
)
affected_runs = pd.DataFrame(
    RunStore(db).get_by_escalation(ESCALATION_ID)
    if ESCALATION_ID is not None
    else []
)
display(affected_runs)

,id,signature,reason,affected_count,snapshot,status,created_at


""


""


## `invalid_target_phrases`

Small lookup table used to recognize pages that are not product targets.

In [8]:
phrases = pd.read_sql_query('SELECT * FROM invalid_target_phrases ORDER BY id DESC', db.conn)
display(phrases)

,id,site,phrase,source,added_at


## Full-field drill-down

Overview tables truncate long code and snapshots. Call `show_blob` with a table, row id, and column to print the complete value; JSON is formatted for readability.

In [9]:
REVIEW_TABLES = {
    'parsers', 'golden_samples', 'scrape_runs', 'results', 'escalations', 'invalid_target_phrases'
}


def show_blob(table: str, row_id: int, column: str) -> None:
    """Print one complete text/JSON field from a reviewed table."""
    if table not in REVIEW_TABLES:
        raise ValueError(f'Unknown review table: {table}')
    valid_columns = {row['name'] for row in db.conn.execute(f'PRAGMA table_info({table})')}
    if column not in valid_columns:
        raise ValueError(f'Unknown column for {table}: {column}')

    row = db.conn.execute(
        f'SELECT {column} FROM {table} WHERE id = ?', (row_id,)
    ).fetchone()
    if row is None:
        raise LookupError(f'No {table} row with id={row_id}')

    value = row[0]
    if value is None:
        print('(NULL)')
        return
    try:
        print(json.dumps(json.loads(value), indent=2, ensure_ascii=False, default=str))
    except (TypeError, json.JSONDecodeError):
        print(value)


# Examples:
# show_blob('parsers', 1, 'code')
# show_blob('golden_samples', 1, 'html_snapshot')
# show_blob('results', 1, 'product_data')
# show_blob('escalations', 1, 'snapshot')

## Handy filtered questions

For common site-level questions, the purpose-built stores are more convenient than writing the aggregation again. Change `SITE` and re-run this cell.

In [10]:
from src.scraping.storage import ParserStore, RunStore

SITE = 'tesco'
active_parsers = pd.DataFrame(ParserStore(db).get_active_ordered_by_hits(SITE))
parser_hit_rates = pd.DataFrame(RunStore(db).get_hit_rates(SITE))

print(f'Active parsers for {SITE}:')
display(active_parsers)
print(f'Parser hit rates for {SITE}:')
display(parser_hit_rates)

Active parsers for tesco:


,id,site,version,code,page_type_scope,status,created_at,created_by,hits
0,3,tesco,cs_20260818_200257,"from bs4 import BeautifulSoup\nimport re\nimport json\n\ndef parse(html: str, url: str) -> dict:\n # Non-product page detection\n path = url.split('?'...",None,active,2026-08-18T20:02:57.330Z,initial,2


Parser hit rates for tesco:


,winning_parser_id,hits
0,3,2


## 清空数据库 (Clear database by site)

⛔  **DESTRUCTIVE**. This is the single entry point for permanently deleting a site's rows from `parsers`, `golden_samples`, `results`, `escalations`, `invalid_target_phrases`, and `scrape_runs`.

The default selection deletes only `parsers` + `golden_samples`. Because `escalations` has no `site` column, it is matched by the exact site segment before the first `|` in `signature`. If `parsers` is selected without `scrape_runs`, retained runs have `winning_parser_id` set to NULL. Selecting `scrape_runs` truly deletes the site's run history and detaches retained `results.run_id` values; selecting `escalations` detaches retained `scrape_runs.escalation_id` values.

This is typically used before re-cold-starting a site. Configure `SITE` and `TABLES`, then run the preview before enabling `CONFIRM`.

### 选择站点与要清除的表

In [11]:
from src.scraping.storage.database import CLEARABLE_TABLES, DEFAULT_CLEAR_TABLES

SITE = "tesco"                       # ← canonical site key
TABLES = list(DEFAULT_CLEAR_TABLES)  # ← 可加 'results' / 'escalations' /
                                     #    'invalid_target_phrases' / 'scrape_runs'

print(f"Clearable tables: {', '.join(CLEARABLE_TABLES)}")
print(f"Selected: site={SITE!r} tables={TABLES}")

Clearable tables: parsers, golden_samples, results, escalations, invalid_target_phrases, scrape_runs
Selected: site='tesco' tables=['parsers', 'golden_samples']


In [12]:
# Preview every clearable table. Escalation sites come from signature prefixes.
unknown_tables = [t for t in TABLES if t not in CLEARABLE_TABLES]
if unknown_tables:
    raise ValueError(
        f"Unknown table(s): {unknown_tables!r}. "
        f"Valid tables: {', '.join(CLEARABLE_TABLES)}"
    )
selected_tables = set(TABLES)
site_column_tables = [t for t in CLEARABLE_TABLES if t != 'escalations']

sites: set[str] = set()
for t in site_column_tables:
    for r in db.conn.execute(f"SELECT DISTINCT site FROM {t}"):
        sites.add(r["site"])
for r in db.conn.execute(
    "SELECT DISTINCT substr(signature, 1, "
    "instr(signature || '|', '|') - 1) AS site FROM escalations"
):
    if r["site"]:
        sites.add(r["site"])

if sites:
    print("Sites in database (canonical keys) and row counts:")
    for site in sorted(sites):
        parts = []
        clear_total = 0
        for t in CLEARABLE_TABLES:
            if t == "escalations":
                n = db.conn.execute(
                    "SELECT COUNT(*) FROM escalations WHERE substr(signature, 1, "
                    "instr(signature || '|', '|') - 1) = ?",
                    (site,),
                ).fetchone()[0]
            else:
                n = db.conn.execute(
                    f"SELECT COUNT(*) FROM {t} WHERE site = ?", (site,)
                ).fetchone()[0]
            marker = " ← WILL BE CLEARED" if t in selected_tables else ""
            parts.append(f"{t}={n}{marker}")
            if t in selected_tables:
                clear_total += n
        print(f"  {site}: {clear_total} rows to clear ({', '.join(parts)})")
        if "parsers" in selected_tables and "scrape_runs" not in selected_tables:
            detached = db.conn.execute(
                "SELECT COUNT(*) FROM scrape_runs "
                "WHERE site = ? AND winning_parser_id IS NOT NULL",
                (site,),
            ).fetchone()[0]
            if detached:
                print(
                    f"    NOTE: {detached} scrape_runs will be retained "
                    "but their parser FK will be set to NULL."
                )
else:
    print("No site-scoped data found, including escalation signatures.")

Sites in database (canonical keys) and row counts:
  amazon: 0 rows to clear (parsers=0 ← WILL BE CLEARED, golden_samples=0 ← WILL BE CLEARED, results=2, escalations=0, invalid_target_phrases=0, scrape_runs=1)
  argos: 5 rows to clear (parsers=1 ← WILL BE CLEARED, golden_samples=4 ← WILL BE CLEARED, results=1, escalations=0, invalid_target_phrases=0, scrape_runs=1)
    NOTE: 1 scrape_runs will be retained but their parser FK will be set to NULL.
  tesco: 10 rows to clear (parsers=1 ← WILL BE CLEARED, golden_samples=9 ← WILL BE CLEARED, results=2, escalations=0, invalid_target_phrases=0, scrape_runs=2)
    NOTE: 2 scrape_runs will be retained but their parser FK will be set to NULL.


In [13]:
CONFIRM = False  # ← set to True to execute the hard delete

if SITE and TABLES and CONFIRM:
    counts = db.clear_site(SITE, tables=TABLES)
    detail = ", ".join(f"{table}={count}" for table, count in counts.items())
    print(f"✓ Site '{SITE}' cleared: {detail}")
else:
    print("Set SITE and TABLES, then CONFIRM = True to clear a site.")

Set SITE and TABLES, then CONFIRM = True to clear a site.
